# Ground states as matrix product states

`MPSGSOperator` is a density operator $\rho = |\psi_0\rangle\langle\psi_0|$ whose state is the ground state of a Hamiltonian, stored as a matrix product state (MPS) computed with a DMRG. It gives `expect` and `partial_trace` for systems that are too large for exact diagonalization.

This notebook shows how to use it, how to choose the order of the sites along the chain, how to check the accuracy, and how to compare it with Gibbs states at low temperature. See the user guide (*Ground states as matrix product states*) for the details and limitations.

> The MPS ground-state operator is experimental, and the ground state is assumed to be non-degenerate.

In [ ]:
import time

import numpy as np
from scipy.sparse.linalg import eigsh

from qalma import build_system
from qalma.mps import MPS, MPSGSOperator
from qalma.operators.states import GibbsDensityOperator

## 1. A small chain, checked against exact diagonalization

The ground state of the Heisenberg chain with open boundaries and 12 spins. The argument `permutation` gives the order of the sites along the chain; here we list the names explicitly. The DMRG runs the first time the energy is needed.

In [ ]:
n_sites = 12
system = build_system("open chain lattice", "spin", L=n_sites)
H = system.global_operator("Hamiltonian")
order = [f"1[{i}]" for i in range(n_sites)]

rho = MPSGSOperator(H, permutation=order, chi_max=32)
print("DMRG energy:", rho.energy)
print(rho.dmrg_info["converged"], rho.dmrg_info["sweeps"], rho.dmrg_info["max_truncation"])
print(rho)

In [ ]:
block = tuple(sorted(system.sites))
e_exact = eigsh(H.to_qutip(block).data_as("csr_matrix"), k=1, which="SA")[0][0]
print("exact energy:", e_exact, " difference:", rho.energy - e_exact)

Expectation values of operators of any support are computed from the MPS, with no dense matrix of the full system. The cost of the reduced state of a few sites is exponential in the number of sites kept, but only linear in the distance between them along the chain.

In [ ]:
Sz = lambda i: system.site_operator("Sz", f"1[{i}]")

for r in range(1, 6):
    print(r, np.real(rho.expect(Sz(0) * Sz(r))))

In [ ]:
reduced = rho.partial_trace(frozenset({"1[5]", "1[6]"}))
print(type(reduced).__name__)
print(reduced.to_qutip(("1[5]", "1[6]")))

## 2. Beyond exact diagonalization

An open chain of 40 spins has $2^{40}$ states. The ground state with bond dimension 32 takes a few seconds.

In [ ]:
n_big = 40
big = build_system("open chain lattice", "spin", L=n_big)
H_big = big.global_operator("Hamiltonian")
natural = [f"1[{i}]" for i in range(n_big)]

start = time.time()
rho_big = MPSGSOperator(H_big, permutation=natural, chi_max=32)
print("E0 =", rho_big.energy, " E0/N =", rho_big.energy / n_big)
print("converged:", rho_big.dmrg_info["converged"],
      " sweeps:", rho_big.dmrg_info["sweeps"],
      " truncation:", rho_big.dmrg_info["max_truncation"])
print("time: %.1f s" % (time.time() - start))

In [ ]:
# Spin-spin correlations from the middle of the chain
center = n_big // 2
for r in (1, 2, 3, 4, 8, 16):
    value = rho_big.expect(big.site_operator("Sz", f"1[{center}]")
                           * big.site_operator("Sz", f"1[{center + r}]"))
    print(r, np.real(value))

## 3. The order of the sites matters

By default the chain follows the lexicographic order of the site names, which for a chain with more than ten sites is `1[0], 1[10], 1[11], ..., 1[1], 1[2], ...`. The state is the same if the bond dimension is large enough, but the DMRG needs more sweeps, a larger truncation and more time. Here, with 20 spins and the same bond dimension:

In [ ]:
n_ord = 20
sys_ord = build_system("open chain lattice", "spin", L=n_ord)
H_ord = sys_ord.global_operator("Hamiltonian")

for label, perm in (("natural", [f"1[{i}]" for i in range(n_ord)]),
                    ("lexicographic", None)):
    start = time.time()
    state = MPSGSOperator(H_ord, permutation=perm, chi_max=32,
                          dmrg_options={"max_sweeps": 10})
    energy = state.energy
    info = state.dmrg_info
    print(f"{label:14s} E0 = {energy:.8f}  sweeps = {info['sweeps']:2d}  "
          f"converged = {info['converged']}  truncation = {info['max_truncation']:.1e}  "
          f"({time.time() - start:.1f} s)")

## 4. A two-dimensional lattice

For a lattice of $4\times3$ spins with open boundaries, the sites are named `1[x, y]`. The lexicographic order runs along $x$ in blocks of three sites; the *snake* runs over rows of four sites. With a bond dimension of 64, which is exact for 12 spins, both give the exact energy.

In [ ]:
lattice = build_system("open square lattice", "spin", L=4, W=3)
H_lat = lattice.global_operator("Hamiltonian")
lex = tuple(sorted(lattice.sites))
coords = {tuple(int(v) for v in name[2:-1].split(",")): name for name in lex}
snake = [coords[(x if y % 2 == 0 else 3 - x, y)] for y in range(3) for x in range(4)]

e_exact = eigsh(H_lat.to_qutip(lex).data_as("csr_matrix"), k=1, which="SA")[0][0]
print("exact:", e_exact)
for label, perm in (("lexicographic", None), ("snake", snake)):
    state = MPSGSOperator(H_lat, permutation=perm, chi_max=64)
    print(f"{label:14s} E0 = {state.energy:.8f}   order starts with {state.order[:4]}")

## 5. Checking convergence with the bond dimension

The truncation error and the energy have to be checked against `chi_max`. Here, for the same lattice and the lexicographic order:

In [ ]:
for chi in (8, 16, 32, 64):
    state = MPSGSOperator(H_lat, chi_max=chi)
    print(f"chi_max = {chi:2d}   E0 - exact = {state.energy - e_exact: .1e}   "
          f"truncation = {state.dmrg_info['max_truncation']:.1e}   "
          f"converged = {state.dmrg_info['converged']}")

## 6. Gibbs states at low temperature

The reduced state of a Gibbs state tends to the one of the ground state when $\beta\to\infty$. For the bond term $S^z_3 S^z_4$ of an open chain with 8 spins, the exact Gibbs states (dense) and the MPS give:

In [ ]:
n_g = 8
sys_g = build_system("open chain lattice", "spin", L=n_g)
H_g = sys_g.global_operator("Hamiltonian")
obs = sys_g.site_operator("Sz", "1[3]") * sys_g.site_operator("Sz", "1[4]")
sites = frozenset({"1[3]", "1[4]"})

gs = MPSGSOperator(H_g, permutation=[f"1[{i}]" for i in range(n_g)])
print("ground state:", np.real(gs.partial_trace(sites).expect(obs)))
for beta in (1, 2, 4, 8, 16, 32):
    gibbs = GibbsDensityOperator((beta * H_g).simplify())
    print(f"beta = {beta:2d}:", np.real(gibbs.partial_trace(sites).expect(obs)))

## 7. Using an MPS computed elsewhere

The DMRG of QALMA is plain. A state computed with another code can be wrapped with `from_mps`, as long as its positions follow the same `permutation`. As an illustration, we build the MPS of the exact ground state of the 8-site chain from its state vector.

In [ ]:
block = tuple(sorted(sys_g.sites))          # '1[0]', ..., '1[7]': the natural order
vals, vecs = eigsh(H_g.to_qutip(block).data_as("csr_matrix"), k=1, which="SA")
mps = MPS.from_dense(vecs[:, 0], [2] * n_g)
print(mps, mps.bond_dims)

wrapped = MPSGSOperator.from_mps(mps, sys_g, permutation=list(block), hamiltonian=H_g)
print(wrapped.expect(H_g), vals[0])

## 8. Saving and loading

The state is stored in HDF5 files with the tensors of the MPS, the order of the sites and the Hamiltonian. No dense matrix is written and nothing is stored as a pickle.

In [ ]:
import os
import tempfile

from qalma.io import load_operators, save_operators

filename = os.path.join(tempfile.mkdtemp(), "ground_state.h5")
save_operators(filename, {"rho": gs, "H": H_g})
print(os.path.getsize(filename), "bytes")

loaded = load_operators(filename)
rho_loaded = loaded["rho"]
print(rho_loaded, "- solved:", rho_loaded.solved)

loaded_system = rho_loaded.system
obs_loaded = loaded_system.site_operator("Sz", "1[3]") * loaded_system.site_operator("Sz", "1[4]")
print(np.real(rho_loaded.partial_trace(sites).expect(obs_loaded)))